# Chapter 00: Essentials: YAML, Git Refs, CI/CD Vocabulary (notebook edition)

## Learning Objectives

- Reproduce both YAML traps
- Normalize the `on` key
- Walk a commit chain from a ref

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Parse Tally's workflow

We load the workflow text with PyYAML to see exactly what a parser sees. This matters because every later tool we write reads workflow files this way. You should see a dict whose first key is `True`.

In [ ]:
import yaml
TALLY = """
name: CI
on: push
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: [3.9, 3.10, 3.11]
    steps:
      - uses: actions/checkout@v4
      - run: python -m pytest
"""
doc = yaml.safe_load(TALLY)
print(list(doc.keys()))
assert True in doc and "on" not in doc

## 2. Trap 1: the float version

We read the matrix versions back out. `3.10` should have become `3.1`; quoting should preserve it.

In [ ]:
versions = doc["jobs"]["test"]["strategy"]["matrix"]["python-version"]
print(versions)
assert versions == [3.9, 3.1, 3.11]
quoted = yaml.safe_load("v: ['3.9', '3.10', '3.11']")["v"]
print(quoted)
assert quoted == ["3.9", "3.10", "3.11"]

## 3. Trap 2: normalize `on`

Our helper maps the boolean key back to `"on"` so downstream code can use the real name. You should see `{'push': {}}`.

In [ ]:
from intro_gha.workflow import triggers
doc["on"] = doc.pop(True)
print(triggers(doc))
assert triggers(doc) == {"push": {}}

## 4. Walk Tally's Git history

Refs are names for commits; parents form a chain. We walk it from `main`.

In [ ]:
HISTORY = {"a1b2c3d": None, "e4f5a6b": "a1b2c3d", "9c8d7e6": "e4f5a6b"}
chain = ["9c8d7e6"]
while HISTORY[chain[-1]]:
    chain.append(HISTORY[chain[-1]])
print(" <- ".join(reversed(chain)))
assert chain == ["9c8d7e6", "e4f5a6b", "a1b2c3d"]

## Takeaways & Next Steps

- Quote versions; `3.10` is the float `3.1`.
- `on` is `True` to a YAML 1.1 parser; normalize it.
- Next: Chapter 01, why Actions at all.

## Chapter Link

Read: `learning_modules/chapter_00_essentials_yaml_git_cicd.md`